# 00 - data audit, `china`

Coverage, missingness, dtypes, units, vintages, duplicates and the province list, for the
provincial-year panel this project rests on.

**There is no inference in this notebook and there must never be any.** Nothing here fits a
model, tests a hypothesis or computes a detection statistic. No provincial sum is compared
with a national total, no growth rate is derived from a level, no proxy is regressed on
anything. Its whole job is to say what is on disk, which province and year cells exist per
series and per vintage, and in which units, so that the tests in `src/china/analysis/` can
later be run against something known. The confounds are in `docs/known_traps.md`; the
acceptance checks an extraction has to reproduce before its numbers are used are in section 6
of `docs/data_dictionary.md`.

If `data/raw` is empty, every cell that reads data prints what to run and does nothing else.
The catalogue cells (the series list, the province list) describe code rather than data and
run either way.

In [ ]:
from __future__ import annotations

from pathlib import Path

import china
import pandas as pd
from china.acquire.yearbook import (
    EDITION_YEARS,
    ELECTRICITY_TITLE,
    FREIGHT_TITLES,
    GRP_TITLE,
)
from china.clean.pbc_reports import parse_summary_links
from china.clean.provinces import (
    BOUNDARY_CHANGES,
    NON_PROVINCE_ROWS,
    PROVINCES,
    REBASING_YEARS,
    SUBPROVINCIAL_UNITS,
    canonical_province,
)
from china.clean.schema import (
    NATIONAL,
    PANEL_COLUMNS,
    PANEL_DTYPES,
    PANEL_KEY,
    SERIES,
    coerce_panel,
    concat_vintages,
    empty_panel,
    validate_panel,
)
from china.clean.worldbank import INDICATOR_TO_SERIES, parse_indicator_response, to_panel
from china.clean.yearbook import find_tables, parse_toc
from forensics_core.provenance import load_sources

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)

PROJECT = Path(china.__file__).resolve().parents[2]
DATA = PROJECT / "data"
RAW = DATA / "raw"

RUN_FIRST = (
    "data/raw is empty. From projects/china run:\n"
    "    make data     # acquire the sources listed in data/SOURCES.yaml\n"
    "and then re-run this notebook. Nothing is computed until then."
)

RAW_FILES = (
    sorted(p for p in RAW.rglob("*") if p.is_file() and p.name != ".gitkeep")
    if RAW.is_dir()
    else []
)
RAW_EMPTY = not RAW_FILES
if RAW_EMPTY:
    print(RUN_FIRST)

## The bureau's portal is unreachable from this network

This has to be read before any coverage number below is interpreted, because it decides what
coverage can even mean here.

Every request to the National Bureau of Statistics data portal at
`data.stats.gov.cn/easyquery.htm` returned **HTTP 403** from the site's web application
firewall, in Chinese and in English, reproduced across every combination of user agent, cookie
jar and header, from two different egress addresses and through two different tools. The
sibling legacy paths return HTTP 404 with an application error body, so the legacy API is
retired rather than merely firewalled. `data/ACCESS_NOTES.md` records each failure verbatim
and `src/china/acquire/` deliberately ships **no acquirer** for any of those three registry
entries.

So the panel depends on the fallbacks, and each of them is partial:

- **The yearbook editions**, which are reachable and are the better source anyway, because
  each edition is a frozen snapshot that was never retro-revised and the archive of editions
  therefore *is* the vintage series this project needs. But the 2024 edition's contents frame
  holds 762 links, 702 of them `.jpg` and not one a spreadsheet. **The provincial tables are
  photographs.** `china.clean.yearbook.extract_table_image` is a stub that raises, and it must
  not be replaced by a plausible-looking parser.
- **The World Bank indicator API**, which is machine-readable but carries the national figure
  only, and only in the current vintage. Mixing it with provincial rows from a yearbook
  edition fabricates a gap that is really a revision.
- **The central bank's regional reports**, which publish loan balances as rounded prose.

The consequence for this audit: **no provincial number can be read by any code in this tree
today.** A provincial coverage grid that comes back empty below is reporting that fact, not a
failure of the notebook.

In [ ]:
SOURCES = load_sources(DATA / "SOURCES.yaml")
registry = pd.DataFrame(
    [
        {
            "id": s.id,
            "access": s.access,
            "status": s.status,
            "verdict": (s.verification or {}).get("verdict", "not_verified"),
            "blocked_reason": (s.blocked_reason or "")[:80],
            "local_path": s.local_path or "",
            "on_disk": bool(s.local_path) and (PROJECT / s.local_path).exists(),
        }
        for s in SOURCES
    ]
)
print(f"{len(registry)} registry entries in data/SOURCES.yaml")
display(pd.crosstab(registry["status"], registry["access"], margins=True))
display(registry["verdict"].value_counts().rename("entries").to_frame())
print("blocked entries:")
display(registry.loc[registry["status"] == "blocked", ["id", "blocked_reason"]])
spine = registry.loc[registry["id"].str.startswith("csy_"), ["id", "status", "verdict"]]
print("the yearbook family the panel depends on:")
display(spine)
acquired = registry.loc[registry["local_path"].ne("")]
print(f"{len(acquired)} entries record a local_path; {int(acquired['on_disk'].sum())} are on disk")

In [ ]:
if RAW_EMPTY:
    print(RUN_FIRST)
else:
    inventory = pd.DataFrame(
        [
            {
                "directory": p.parent.relative_to(RAW).as_posix() or ".",
                "suffix": p.suffix.lower(),
                "bytes": p.stat().st_size,
            }
            for p in RAW_FILES
        ]
    )
    display(
        inventory.groupby("directory")
        .agg(n_files=("bytes", "size"), total_bytes=("bytes", "sum"))
        .sort_index()
    )
    display(
        inventory.groupby("suffix")
        .agg(n_files=("bytes", "size"), total_bytes=("bytes", "sum"))
        .sort_values("n_files", ascending=False)
    )
    print(f"{len(RAW_FILES):,} files, {int(inventory['bytes'].sum()):,} bytes under data/raw")

## The yearbook editions, and what form their tables take

The index lists editions from 2005 to 2025; earlier ones exist only as complete-volume
archives on the Wayback Machine. A table is located by its printed **title** and never by a
guessed file name, because the file-naming convention changed between editions (2017
`html/EN0309.jpg`, 2020 `html/E0309.jpg`, 2024 `html/E03-09.jpg`) and the registry warns that
the table number drifts too. `parse_toc` keeps both the printed number and the number implied
by the file name so that a disagreement stays visible instead of being resolved silently.

The `kind` column is the one that matters. An edition whose regional tables are all `jpg`
needs optical character recognition; one with `htm` tables does not, and that is worth
checking edition by edition rather than assuming.

In [ ]:
CSY = RAW / "csy"
TOC_PATHS = sorted(CSY.glob("*/toc_*.htm")) if CSY.is_dir() else []
EDITIONS = None
YEARBOOK_TABLES = None

if not TOC_PATHS:
    print(RUN_FIRST if RAW_EMPTY else "no contents frames under data/raw/csy")
else:
    wanted = (GRP_TITLE, ELECTRICITY_TITLE, *FREIGHT_TITLES)
    edition_rows, table_rows = [], []
    for path in TOC_PATHS:
        edition = path.parent.name
        language = path.stem.rsplit("_", 1)[-1]
        try:
            toc = parse_toc(path.read_bytes())
        except ValueError as exc:
            print(f"{edition}/{path.name}: not a contents frame ({exc})")
            continue
        kinds = toc["kind"].value_counts()
        edition_rows.append(
            {
                "edition": edition,
                "language": language,
                "links": len(toc),
                "jpg": int(kinds.get("jpg", 0)),
                "htm": int(kinds.get("htm", 0)) + int(kinds.get("html", 0)),
                "pdf": int(kinds.get("pdf", 0)),
                "spreadsheet": int(kinds.get("xls", 0)) + int(kinds.get("xlsx", 0)),
                "other": int(kinds.get("other", 0)),
            }
        )
        for title in wanted:
            for row in find_tables(toc, title).itertuples():
                table_rows.append(
                    {
                        "edition": edition,
                        "language": language,
                        "looked_for": title,
                        "table_number": row.table_number,
                        "file_table_number": row.file_table_number,
                        "numbers_agree": row.table_number == row.file_table_number,
                        "kind": row.kind,
                        "data_year": row.data_year,
                        "image_on_disk": (path.parent / "html" / str(row.filename)).is_file(),
                    }
                )
    EDITIONS = pd.DataFrame(edition_rows)
    YEARBOOK_TABLES = pd.DataFrame(table_rows)
    display(EDITIONS.sort_values(["edition", "language"]))
    print(
        f"editions the index lists: {len(EDITION_YEARS)} "
        f"({min(EDITION_YEARS)} to {max(EDITION_YEARS)}); "
        f"editions with a contents frame on disk: {EDITIONS['edition'].nunique()}"
    )
    if len(YEARBOOK_TABLES):
        display(YEARBOOK_TABLES.sort_values(["edition", "looked_for"]))
        display(YEARBOOK_TABLES.groupby(["looked_for", "kind"]).size().rename("menu_entries"))
        print(
            f"table images on disk: {int(YEARBOOK_TABLES['image_on_disk'].sum())} "
            f"of {len(YEARBOOK_TABLES)} menu entries"
        )
    else:
        print("none of the three regional tables was found in any contents frame on disk")

## The panel, and the vintage column it turns on

The panel is `province, year, series, value, unit, vintage, source_id`, keyed on
`(province, year, series, **vintage**)`. Vintage is not bookkeeping. Revisions overwrite
history: the bureau and the provincial bureaus serve the current vintage only, and the
pre-revision Liaoning figures for 2011 to 2014 are not in it. They are in the 2015 edition of
the yearbook, which was never retro-revised. So the same province-year-series legitimately
holds several values, one per vintage, and a panel without the column keeps whichever was
loaded last.

Two rules the validator enforces and this audit therefore reports rather than repairs: a
missing observation is an **absent row, never a NaN row**, and an unrecognised province label
is an **error, not a row to drop**, because it almost always means an extraction went wrong.

The next cell builds whatever panel rows the code in this tree can actually produce from
`data/raw`. Today that is the World Bank national series and nothing else. The central bank
loan balances are parsed but cannot enter the panel yet, for the reason given further down;
the yearbook tables cannot be read at all.

In [ ]:
WORLDBANK = RAW / "worldbank"
wb_files = sorted(WORLDBANK.glob("*.json")) if WORLDBANK.is_dir() else []
parts = []
for path in wb_files:
    try:
        observations = parse_indicator_response(path.read_bytes())
    except ValueError as exc:
        print(f"{path.name}: not an indicator response ({exc})")
        continue
    if observations.empty:
        print(f"{path.name}: no non-null observations")
        continue
    unmapped = sorted({i for i in observations["indicator"] if i not in INDICATOR_TO_SERIES})
    if unmapped:
        print(
            f"{path.name}: no panel series is defined for {unmapped}; acquired as context, "
            f"not loaded"
        )
        continue
    parts.append(coerce_panel(to_panel(observations)))

PANEL = concat_vintages(parts) if parts else empty_panel()
if RAW_EMPTY:
    print(RUN_FIRST)
print(f"panel rows: {len(PANEL):,}; columns {list(PANEL.columns) == list(PANEL_COLUMNS)}")
provincial = PANEL.loc[PANEL["province"] != NATIONAL]
print(f"national rows: {len(PANEL) - len(provincial):,}; provincial rows: {len(provincial):,}")
if provincial.empty:
    print(
        "no provincial rows: every provincial number is inside a JPEG and "
        "china.clean.yearbook.extract_table_image is a stub that raises"
    )
problems = validate_panel(PANEL, known_source_ids={s.id for s in SOURCES})
print("validate_panel problems:", problems if problems else "none")

In [ ]:
if PANEL.empty:
    print("the panel is empty, so there are no vintages to compare")
else:
    display(
        PANEL.groupby(["vintage", "series"]).agg(
            rows=("value", "size"),
            provinces=("province", "nunique"),
            first_year=("year", "min"),
            last_year=("year", "max"),
            source_ids=("source_id", "nunique"),
        )
    )
    duplicated = PANEL.duplicated(subset=list(PANEL_KEY), keep=False)
    print(f"rows duplicating the key {PANEL_KEY}: {int(duplicated.sum()):,}")
    per_cell = PANEL.groupby(["province", "year", "series"])["vintage"].nunique()
    print(
        f"province-year-series cells carried by more than one vintage: "
        f"{int((per_cell > 1).sum()):,} of {len(per_cell):,}"
    )
    print("a cell with two vintages is the measurement, not a conflict")

## The coverage grid, and missingness by series and year

Two views of the same absence. The grid says which province and year cells exist for each
series and vintage; the table after it counts, per series and year, how many of the 31
provincial-level units are present.

Read the denominator carefully. Thirty-one is the number of rows the yearbook's regional
tables carry, and it is not a target every series can meet: electricity is printed for
selected years only rather than annually, freight is one cross-section per edition, and
provincial credit is not published as a table anywhere free, which the registry establishes
three independent ways. A gap in the grid can be a publication decision, an acquisition that
has not run, or an extraction that cannot run. This audit does not distinguish them; it only
shows where they are.

In [ ]:
if PANEL.empty:
    print("the panel is empty, so there is no coverage grid")
else:
    for (series_name, vintage), group in PANEL.groupby(["series", "vintage"]):
        grid = group.pivot_table(
            index="province", columns="year", values="value", aggfunc="size", fill_value=0
        )
        print(
            f"--- {series_name} / {vintage}: {grid.shape[0]} row labels, "
            f"{grid.shape[1]} years, {int(group['value'].size)} cells"
        )
        display(grid)

In [ ]:
if PANEL.empty:
    print("the panel is empty, so there is nothing to count as missing")
else:
    expected = len(PROVINCES)
    rows = []
    for (series_name, year), group in provincial.groupby(["series", "year"]):
        present = int(group["province"].nunique())
        rows.append(
            {
                "series": series_name,
                "year": int(year),
                "provinces_present": present,
                "provinces_absent": expected - present,
                "share_present": present / expected,
            }
        )
    missingness = pd.DataFrame(rows)
    if len(missingness):
        display(missingness.sort_values(["series", "year"]))
    else:
        print(
            f"no provincial rows at all, so all {expected} provinces are absent in every "
            f"series and year"
        )
    national = PANEL.loc[PANEL["province"] == NATIONAL]
    if len(national):
        display(
            national.groupby(["series", "vintage"]).agg(
                years=("year", "nunique"), first_year=("year", "min"), last_year=("year", "max")
            )
        )

## Units, price bases, and the deflator that does not exist

Units are as printed by the publisher, and conversion happens once, in the loader, never
downstream. Money is **100 million yuan** throughout because that is what the yearbook prints;
the World Bank prints yuan, so its loader divides by 1e8 and nothing else converts anything.
The central bank publishes trillions and hundreds of millions within a single sentence, and
its parser keeps the printed figure and its printed unit beside the converted value so the
rounding stays visible.

Three things this notebook cannot check and must therefore state:

1. **Nominal levels and real growth are different quantities.** `grp_nominal` is at current
   prices; `grp_index_preceding_year` is at constant prices with the preceding year as 100,
   and it is *not* the growth rate of `grp_nominal`. The two behave differently and must never
   be mixed.
2. **The provincial index is at provincial deflators** while the national growth rate is at
   national ones, and **there is no provincial deflator series in the registry.** That is why
   `china.analysis.gap.mechanical_gap_components` refuses to run and why the residual can only
   be bounded, not decomposed.
3. **One declared unit is unconfirmed.** `freight_ton_km` carries the literal `TO CONFIRM`
   because table 16-15 was listed in two contents frames but never opened.

In [ ]:
catalogue = pd.DataFrame(
    [
        {
            "series": spec.name,
            "unit": spec.unit,
            "unit_confirmed": spec.unit_confirmed,
            "n_source_ids": len(spec.source_ids),
            "source_ids": ";".join(spec.source_ids),
        }
        for spec in SERIES.values()
    ]
)
display(catalogue)
print("units never read off a table:", [s.name for s in SERIES.values() if not s.unit_confirmed])
print("series with no verified free source:", [s.name for s in SERIES.values() if not s.source_ids])
display(pd.DataFrame({"declared": pd.Series(PANEL_DTYPES), "actual": PANEL.dtypes.astype(str)}))
if not PANEL.empty:
    used = PANEL.groupby("series")["unit"].agg(lambda u: sorted(set(u)))
    display(used.rename("units_in_the_panel").to_frame())
    wrong = [name for name, units in used.items() if list(units) != [SERIES[name].unit]]
    print("series carrying a unit other than the declared one:", wrong)

## The province list: names that drift, rows that are not provinces, revisions below the line

Three hazards, each of which silently corrupts a provincial sum.

**Names drift between editions.** The 2015 edition labels one row `Tibet`; the 2024 edition
labels the same row `Xizang`. Stack two vintages without a name map and one province becomes
two. Only that alias is attested in `data/SOURCES.yaml`; the rest of the alias table is the
conventional form and is marked TO CONFIRM in `src/china/clean/provinces.py` until it has been
seen in an extracted table. `Shanxi` and `Shaanxi` are different provinces and the table keeps
them apart explicitly.

**Some rows are not provinces.** Table 16-14 carries a `National Total` row and a
`Not Classified by Region` residual row, the latter being civil aviation and pipelines.
Summing the column as printed double counts the national total and adds the residual on top.

**The admitted revisions happened below the provincial level.** The January 2018 Tianjin
revision was of Binhai New Area, a sub-provincial development zone, so a provincial series
absorbs only part of it; the Inner Mongolia episode named the city of Baotou; and the central
bank publishes 32 report summaries, the 31 provincial units plus Shenzhen, so a naive
file-per-province mapping produces 32 provinces.

**And boundaries changed.** Hainan separated from Guangdong in 1988 and Chongqing from Sichuan
in 1997; the 2004, 2008, 2013 and 2018 economic censuses rebase and back-revise both national
and provincial series. A series that crosses one of these is two series: test around them, not
across them.

In [ ]:
print(f"{len(PROVINCES)} provincial-level units, reserved national label {NATIONAL!r}")
print(", ".join(PROVINCES))
print()
print("aggregate and residual row labels to drop before summing:", sorted(NON_PROVINCE_ROWS))
display(pd.DataFrame([{"unit": u.name, "parent": u.parent} for u in SUBPROVINCIAL_UNITS]))
display(
    pd.DataFrame(
        [
            {"year": b.year, "created": b.created, "from_parent": b.from_parent}
            for b in BOUNDARY_CHANGES
        ]
    )
)
print("economic-census rebasing years:", list(REBASING_YEARS))
if PANEL.empty:
    print("the panel is empty, so no row label has been tested against the list")
else:
    labels = sorted({str(v) for v in PANEL["province"]})
    unrecognised = [x for x in labels if x != NATIONAL and canonical_province(x) is None]
    print(f"distinct province labels in the panel: {len(labels)}")
    print("labels that map to nothing (report these, never drop them):", unrecognised)
    missing_provinces = [p for p in PROVINCES if p not in set(labels)]
    print(f"provinces absent from the panel entirely: {len(missing_provinces)}")
    print(missing_provinces)

## The central bank's regional reports

Free, reachable, and the only provincial credit figures published anywhere without a
subscription, but they arrive as prose rounded to roughly two significant figures, there is no
2016 edition, and the channel publishes 32 summaries rather than 31.

They cannot enter the panel yet, and the reason is recorded rather than worked around:
`china.clean.pbc_reports.map_province_names` takes the Chinese-to-canonical mapping as an
argument and refuses to guess one, because no Chinese-to-English province table in this
project was read from a source. Until that table is built from a fetched report page and
recorded, `to_panel` has no canonical province name to stamp on a row. The cell below counts
what is on disk and what the year pages advertise; it maps nothing.

In [ ]:
PBC = RAW / "pbc"
year_pages = sorted(PBC.glob("*/index.html")) if PBC.is_dir() else []
if not year_pages:
    print(RUN_FIRST if RAW_EMPTY else "no year pages under data/raw/pbc")
else:
    rows = []
    for path in year_pages:
        links = parse_summary_links(path.read_bytes())
        is_summary = links["is_summary"].fillna(False)
        rows.append(
            {
                "report_year": path.parent.name,
                "pdf_links": len(links),
                "summaries": int(is_summary.sum()),
                "with_a_published_name": int((links["province_zh"].astype(str) != "").sum()),
                "pdfs_on_disk": len(list(path.parent.glob("*.pdf"))),
            }
        )
    pbc_coverage = pd.DataFrame(rows).sort_values("report_year")
    display(pbc_coverage)
    print(
        f"report years on disk: {len(pbc_coverage)}; "
        f"summary links: {int(pbc_coverage['summaries'].sum()):,}; "
        f"PDFs on disk: {int(pbc_coverage['pdfs_on_disk'].sum()):,}"
    )
    print(
        "31 provincial units plus Shenzhen is 32 summaries per year; a file-per-province "
        "mapping would produce 32 provinces"
    )

## What this audit could not check

Stated so that nobody mistakes a clean run for a usable panel.

1. **Every provincial number.** They are inside JPEG scans.
   `china.clean.yearbook.extract_table_image` raises, on purpose, and no coverage figure above
   can improve until an extraction back end is chosen and gated on the arithmetic checks in
   section 6 of `docs/data_dictionary.md`.
2. **The alias table.** Only `Tibet` to `Xizang` is attested in `data/SOURCES.yaml`. The other
   five entries are conventional forms that have never been seen in an extracted table, so the
   province-label check above cannot fire on the cases most likely to be wrong.
3. **The Chinese-to-canonical province mapping** for the central bank reports, which does not
   exist and which this notebook deliberately does not invent.
4. **Whether two editions agree** on a data year they both print. That is the cross-edition
   check the data dictionary requires, and it needs at least two extracted tables, of which
   there are currently none.
5. **Anything about the gap.** No provincial sum, no national comparison, no growth rate, no
   proxy relationship and no discontinuity appears above, and none should be added here.